<a href="https://colab.research.google.com/github/Victor-Tung/CS171_news_classification/blob/main/CS171_news_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Data Investigation


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Read the features file.
train_data = pd.read_csv('/content/drive/MyDrive/CS171/ag_train.csv',header=None,
                   names=['class', 'title', 'description'])
test_data = pd.read_csv('/content/drive/MyDrive/CS171/ag_test.csv',header=None,
                   names=['class', 'title', 'description'])
#first 30 lines are all class index 3, data set is well balanced
print(train_data['class'].value_counts().sort_index())
print('-----------------------------------')
train_data.tail(5)


In [ ]:
train_data.shape

In [ ]:
train_data.columns

In [ ]:
train_data.dtypes

# Data Preprocessing and Cleaning


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)        # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()          # collapse extra whitespace
    text = ' '.join([w for w in text.split() if w not in stop_words])  # remove stopwords
    text = re.sub(r"\b(reuters|afp|ap|forbes|cnn|bbc|bloomberg)\b", "", text, flags=re.IGNORECASE)
    return text

In [ ]:
train_data['title_clean'] = train_data['title'].apply(clean_text)
train_data['description_clean'] = train_data['description'].apply(clean_text)
train_data['text'] = train_data['title_clean'] + ' ' + train_data['description_clean']

In [ ]:
test_data['title_clean'] = test_data['title'].apply(clean_text)
test_data['description_clean'] = test_data['description'].apply(clean_text)
test_data['text'] = test_data['title_clean'] + ' ' + test_data['description_clean']

In [ ]:
# testing clean function
test_val = 5000
print("BEFORE:", train_data['title'].iloc[test_val])
print("AFTER: ", train_data['title_clean'].iloc[test_val])
print()
print("BEFORE:", train_data['description'].iloc[test_val])
print("AFTER: ", train_data['description_clean'].iloc[test_val])
train_data.head(5)

# Building/Training Custom CNN


In [ ]:
train_data = train_data[train_data["class"] != "Class Index"].copy()
test_data = test_data[test_data["class"] != "Class Index"].copy()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Raw data
train_texts = train_data["text"].astype(str).values
train_labels = train_data["class"].astype(str).values  # keep as strings

test_texts = test_data["text"].astype(str).values
test_labels = test_data["class"].astype(str).values

# Map class IDs → names
class_names = {
    "1": "World",
    "2": "Sports",
    "3": "Business",
    "4": "Sci/Tech"
}

# Downsample training set
train_texts, _, train_labels, _ = train_test_split(
    train_texts,
    train_labels,
    # train_size=50000,
    random_state=42,
    stratify=train_labels
)

# Encode labels ("1"–"4" → 0–3)
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(train_labels)
y_test_int = label_encoder.transform(test_labels)

# One-hot encode
num_classes = len(label_encoder.classes_)
y_train = to_categorical(y_train_int, num_classes=num_classes)
y_test = to_categorical(y_test_int, num_classes=num_classes)

# Tokenizer
max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

X_train_seq = tokenizer.texts_to_sequences(train_texts)
X_test_seq = tokenizer.texts_to_sequences(test_texts)

X_train = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

print("LabelEncoder classes:", label_encoder.classes_)
print("Class mapping:", class_names)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, SpatialDropout1D
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


embedding_dim = 128
num_classes = y_train.shape[1]

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim),
    Conv1D(filters=128, kernel_size=3, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(64, activation="relu",kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])
model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Train only on train.csv
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

# Test only on test.csv
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Predictions on test.csv
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Convert encoded labels back to original IDs: "1", "2", "3", "4"
predicted_class_ids = label_encoder.inverse_transform(y_pred)
true_class_ids = label_encoder.inverse_transform(y_true)

# Convert IDs to readable class names
predicted_labels = [class_names[i] for i in predicted_class_ids]
true_labels = [class_names[i] for i in true_class_ids]

class_label_names = list(class_names.values())

print("\nClassification Report:\n")
print(classification_report(
    true_labels,
    predicted_labels,
    labels=class_label_names,
    digits=4
))

cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=class_label_names
)

print("\nConfusion Matrix:\n")
print(cm)

cm_normalized = cm.astype("float") / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_label_names,
    yticklabels=class_label_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix on test.csv")
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D,
    Conv1D, GlobalMaxPooling1D, Concatenate,
    Dense, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

embedding_dim = 128
num_classes = y_train.shape[1]

input_layer = Input(shape=(max_len,))

x = Embedding(
    input_dim=max_words,
    output_dim=embedding_dim
)(input_layer)

x = SpatialDropout1D(0.2)(x)

# --- Parallel convolution blocks ---
conv3 = Conv1D(32, kernel_size=3, activation="relu")(x)
conv5 = Conv1D(32, kernel_size=5, activation="relu")(x)
conv7 = Conv1D(32, kernel_size=7, activation="relu")(x)

pool3 = GlobalMaxPooling1D()(conv3)
pool5 = GlobalMaxPooling1D()(conv5)
pool7 = GlobalMaxPooling1D()(conv7)

# Combine all features
x = Concatenate()([pool3, pool5, pool7])

# Dense layers
x = Dense(32, activation="relu")(x)
x = Dropout(0.5)(x)

output_layer = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=input_layer, outputs=output_layer)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Early stopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()

# Test only on test.csv
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Predictions on test.csv
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Convert encoded labels back to original IDs: "1", "2", "3", "4"
predicted_class_ids = label_encoder.inverse_transform(y_pred)
true_class_ids = label_encoder.inverse_transform(y_true)

# Convert IDs to readable class names
predicted_labels = [class_names[i] for i in predicted_class_ids]
true_labels = [class_names[i] for i in true_class_ids]

class_label_names = list(class_names.values())

print("\nClassification Report:\n")
print(classification_report(
    true_labels,
    predicted_labels,
    labels=class_label_names,
    digits=4
))

cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=class_label_names
)

print("\nConfusion Matrix:\n")
print(cm)

cm_normalized = cm.astype("float") / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_label_names,
    yticklabels=class_label_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix on test.csv")
plt.tight_layout()
plt.show()

# Fine-tuning DistilBERT

Install and Imports


In [ ]:
!pip install transformers accelerate -q

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_linear_schedule_with_warmup

from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import pandas as pd
from sklearn.preprocessing import LabelEncoder

train_data = pd.read_csv('/content/drive/MyDrive/CS171/ag_train.csv', header=None,
                         names=['class', 'title', 'description'])
test_data  = pd.read_csv('/content/drive/MyDrive/CS171/ag_test.csv', header=None,
                         names=['class', 'title', 'description'])

train_data = train_data[train_data["class"] != "Class Index"].copy()
test_data  = test_data[test_data["class"] != "Class Index"].copy()

class_names = {"1": "World", "2": "Sports", "3": "Business", "4": "Sci/Tech"}
num_classes = 4

label_encoder = LabelEncoder()
label_encoder.fit(train_data["class"].astype(str).values)

Prepare Raw Text

In [ ]:
train_data_clean = train_data[train_data["class"] != "Class Index"].copy()
test_data_clean  = test_data[test_data["class"] != "Class Index"].copy()

distil_train_texts = (
    train_data_clean["title"].fillna("") + ". " + train_data_clean["description"].fillna("")
).astype(str).values

distil_test_texts = (
    test_data_clean["title"].fillna("") + ". " + test_data_clean["description"].fillna("")
).astype(str).values

from sklearn.model_selection import train_test_split

distil_train_texts, _, distil_train_labels, _ = train_test_split(
    distil_train_texts,
    train_data_clean["class"].astype(str).values,
    train_size=50000,
    random_state=42,
    stratify=train_data_clean["class"].astype(str).values
)
distil_train_texts, distil_val_texts, distil_train_labels, distil_val_labels = train_test_split(
    distil_train_texts,
    distil_train_labels,
    test_size=0.2,
    random_state=42,
    stratify=distil_train_labels
)
distil_test_labels = test_data_clean["class"].astype(str).values

distil_y_train_int = label_encoder.transform(distil_train_labels)
distil_y_val_int   = label_encoder.transform(distil_val_labels)
distil_y_test_int = label_encoder.transform(distil_test_labels)
print(f"Train: {len(distil_train_texts):,} | Val: {len(distil_val_texts):,} | Test: {len(distil_test_texts):,}")

Tokenizer and Dataset

In [ ]:
DISTIL_MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 3e-5

distil_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class AGNewsDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_len):
    self.texts = list(texts)
    self.labels = list(labels)
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)
  def __getitem__(self, idx):
    enc = self.tokenizer(
        self.texts[idx],
        max_length = self.max_len,
        padding = "max_length",
        truncation = True,
        return_tensors = "pt"
    )
    return {
        "input_ids": enc["input_ids"].squeeze(0),
        "attention_mask": enc["attention_mask"].squeeze(0),
        "labels": torch.tensor(self.labels[idx], dtype=torch.long)
    }
train_dataset = AGNewsDataset(distil_train_texts, distil_y_train_int, distil_tokenizer, DISTIL_MAX_LEN)
val_dataset   = AGNewsDataset(distil_val_texts,   distil_y_val_int,   distil_tokenizer, DISTIL_MAX_LEN)
test_dataset = AGNewsDataset(distil_test_texts, distil_y_test_int, distil_tokenizer, DISTIL_MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Training

In [ ]:
distil_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes,
)
distil_model.to(device)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

optimizer = AdamW(distil_model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(distil_model)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
  model.train()
  total_loss, correct, n = 0.0, 0, 0
  for step, batch in enumerate(loader):
    ids = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()
    output = model(input_ids=ids, attention_mask=mask, labels=labels)
    output.loss.backward()
    optimizer.step()
    scheduler.step()

    preds = output.logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total_loss += output.loss.item() * labels.size(0)
    n += labels.size(0)
    if (step + 1 ) % 100 == 0:
      print(f" step {step+1}/{len(loader)} | loss: {total_loss/n:.4f} | acc: {correct/n:.4f}")

  return total_loss / n, correct / n

def evaluate_epoch(model, loader):
  model.eval()
  total_loss, correct, n = 0.0, 0, 0
  with torch.no_grad():
    for batch in loader:
      ids = batch["input_ids"].to(device)
      mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      output = model(input_ids=ids, attention_mask=mask, labels=labels)
      preds = output.logits.argmax(dim=1)
      correct += (preds == labels).sum().item()
      total_loss += output.loss.item() * labels.size(0)
      n += labels.size(0)
  return total_loss / n, correct / n

distil_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
  print(f"\n Epoch {epoch}/ {EPOCHS}")
  train_loss, train_acc = train_epoch(distil_model, train_loader, optimizer, scheduler)
  val_loss, val_acc = evaluate_epoch(distil_model, val_loader)
  distil_history["train_loss"].append(train_loss)
  distil_history["train_acc"].append(train_acc)
  distil_history["val_loss"].append(val_loss)
  distil_history["val_acc"].append(val_acc)
  print(f" Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")
  print(f" Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(distil_history["train_acc"], label="Train Accuracy")
plt.plot(distil_history["val_acc"],   label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("DistilBERT – Training vs Validation Accuracy (3e-5 Learning Rate)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(distil_history["train_loss"], label="Train Loss")
plt.plot(distil_history["val_loss"],   label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DistilBERT – Training vs Validation Loss (3e-5 Learning Rate)")
plt.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CS171/distilbert_faster_learning_rate_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
distil_model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        out  = distil_model(input_ids=ids, attention_mask=mask)
        all_preds.extend(out.logits.argmax(dim=1).cpu().numpy())
        all_true.extend(batch["labels"].numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

test_acc_distil = (all_preds == all_true).mean()
print(f"Test Accuracy (DistilBERT 3e-5 Learning Rate): {test_acc_distil:.4f}")

predicted_class_ids = label_encoder.inverse_transform(all_preds)
true_class_ids      = label_encoder.inverse_transform(all_true)
predicted_labels    = [class_names[i] for i in predicted_class_ids]
true_labels_named   = [class_names[i] for i in true_class_ids]

class_label_names = list(class_names.values())

print("\nClassification Report:\n")
print(classification_report(
    true_labels_named,
    predicted_labels,
    labels=class_label_names,
    digits=4,
))

cm = confusion_matrix(true_labels_named, predicted_labels, labels=class_label_names)
cm_normalized = cm.astype("float") / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_label_names,
    yticklabels=class_label_names,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("DistilBERT (3e-5 Learning Rate) – Normalized Confusion Matrix on test.csv")
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CS171/distilbert_faster_learning_rate_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare/Constrast Results

DistilBERT outperformed both CNN variants on the AG News test set, achieving 93.72% accuracy compared to 91.63% for the simple CNN and 91.08% for the multi-kernel CNN. Surprisingly, the simpler single-kernel CNN slightly outperformed the multi-kernel version, suggesting that parallel convolution branches with different kernel sizes did not add much value for this task.
Looking at per-class performance, Sports was the easiest category for all three models with F1 scores around 0.97-0.98. Business was consistently the hardest, with the CNNs scoring around 0.87-0.88 F1 and DistilBERT reaching 0.91. The most common error across all models was confusing Business and Sci/Tech articles, which makes sense given how much those two categories overlap in real news coverage.

Both CNN models overfit relatively quickly, with validation loss rising after the first couple of epochs while training loss kept falling. DistilBERT showed a similar trend by epoch 3 but still generalized well to the test set. In terms of compute, DistilBERT has around 66 million parameters compared to roughly 2.6 million for the multi-kernel CNN. It required a GPU and several minutes per epoch to train, while the CNNs trained in about 10-15 seconds per epoch. For situations where compute is a concern the CNN is a reasonable option, but where accuracy is the priority DistilBERT is the better choice.